# 在 Google Colab 上玩转 OTTER 生信分析工作流

**什么是 OTTER？**  
OTTER 是一个面向高通量测序分析（RRBS/WGBS 甲基化测序、RNA-seq 转录组测序，以及人鼠嵌合的 PDX/CDX 异种移植肿瘤模型）的一站式生物信息学工作流系统。它能把繁杂的原始测序数据（FASTQ）自动化转换为质控报告、基因表达矩阵以及单碱基分辨率的甲基化定量结果。

在这个 Google Colab 交互式教程中，你不需要在本地配置繁琐的生信集群，只需按顺序运行单元格，即可在 15 分钟内完整体验 OTTER 的现代架构与核心功能：

1. **一键真实安装工具链** —— 下载官方发布的 `otter-install` 安装器，真实部署包括工作流编译器、质控与比对算子在内的全套静态二进制工具，并用自带的 `enva` 搭建好 Conda 运行环境。
2. **理解并模拟参考基因组注册表** —— 了解为什么现代生信工作流需要将“参考基因组”解耦为独立且不可变的注册表（Registry）。我们将在秒级内构建出与生产环境规范 100% 一致的模拟参考基因组。
3. **用真实测序数据创建分析项目 (`otter build`)** —— 使用仓库自带的轻量真实测序数据（Downsampled FASTQ），一条命令自动完成样本识别配对、参考基因组哈希锁定、以及分析任务快照生成。
4. **全流程全阶段预演 (`--dry-run`)** —— 对每个分析场景的**每一个执行阶段（Step 1/2/3 及 Publish）**执行空跑预演，在不真正消耗算力比对的情况下，观察执行器如何把流程编译为完整的任务依赖图（DAG）。

> 💡 **新手必读（本教程不做耗时计算）：**  
> - 这里使用的参考基因组是结构真实的“测试桩（Fixture）”，序列是极短的占位符（因此无需下载几十 GB 的真基因组）；  
> - `--dry-run` 仅运行到调度规划阶段（Planner），不会真正启动耗时数小时的比对计算；  
> - 运行时：**仅需免费 CPU 实例**，无需消耗 GPU 算力。

## 0. 环境准备与系统预检

Colab 会为每个用户分配一台干净的 Linux 云端虚拟机。在正式开始前，我们先检查当前环境是否满足要求（系统架构为 Linux x86_64，且基础系统工具可用）。

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import time


def sh(command, check=True, capture=False, stream=False, echo=None, timeout=None):
    """执行一条 shell 命令，并把命令本身回显出来，让 notebook 读起来像一份操作记录。

    capture=True 会把输出收集起来、在命令结束时一次性打印，适合输出是一整块内容的短命令。
    stream=True 会实时回显输出，多分钟的步骤必须用它：被捕获的长命令在结束前什么都不显示，
    看起来就像卡住了。echo=False 只捕获不打印，用于调用方会自己解析并汇总的输出 —— 一个
    Craftmake plan envelope 是几百 KB 的 JSON，打印出来会把周围的内容全部淹没。
    """
    if echo is not None:
        capture = capture or not stream
    print(f"$ {command}")
    if stream:
        started = time.monotonic()
        process = subprocess.Popen(
            command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        for line in process.stdout:
            print(line.rstrip())
        returncode = process.wait(timeout=timeout)
        print(f"  (exit {returncode} after {time.monotonic() - started:.0f}s)")
        if check and returncode != 0:
            raise subprocess.CalledProcessError(returncode, command)
        return subprocess.CompletedProcess(command, returncode)

    result = subprocess.run(
        command,
        shell=True,
        check=check,
        text=True,
        capture_output=capture or echo is False,
        timeout=timeout,
    )
    if capture and echo is not False and result.stdout.strip():
        print(result.stdout.rstrip())
    return result


print("python  :", sys.version.split()[0])
print("kernel  :", platform.system(), platform.release(), platform.machine())
for tool in ("git", "curl", "tar"):
    print(f"{tool:9}:", shutil.which(tool) or "缺少")

assert platform.system() == "Linux", "OTTER 的发布二进制是 Linux 构建。"
assert platform.machine() in ("x86_64", "amd64"), (
    f"本 notebook 使用 amd64 发布产物，无法在 {platform.machine()} 上运行。"
)
print("\n预检通过。")

### 全局路径与分析场景配置

我们把所有的工作目录、分析场景及参考基因组参数集中配置在此处：
- **测试分析场景**：涵盖常见的四大组学场景 —— RRBS（简化代表性甲基化测序）、RNA-seq（转录组测序）、BS-PDX（人鼠嵌合甲基化模型）、RNA-PDX（人鼠嵌合转录组模型）。
- **测试样本**：来自官方测试集的真实下采样测序数据（SRR 开头的 FASTQ 文件）。
- **工作目录**：`/content/otter-colab`，保存在 Colab 会话的高速临时盘上。

In [ ]:
from pathlib import Path

# 承载 release 与测试 fixture 的公开仓库。
REPO = "otterlab-bio/otter"
RELEASE_TAG = "latest"          # 可固定为 "v1.1.0" 之类，以获得可复现的运行

# 所有内容的落地位置。/content 是 Colab 会话期间持久化的卷；
# /tmp 由内存支撑，跨 cell 会丢失。
WORK = Path("/content/otter-colab")
INSTALL_DIR = WORK / "bin"       # 存放已发布的二进制
REPO_DIR = WORK / "repo"         # 浅克隆，用于 fixture 与 e2e 脚本
REGISTRY = WORK / "registry"     # 模拟的参考基因组注册表
PROJECTS = WORK / "projects"     # 每个场景一个已授权项目

for directory in (WORK, INSTALL_DIR, REGISTRY, PROJECTS):
    directory.mkdir(parents=True, exist_ok=True)

print("仓库      :", REPO)
print("release   :", RELEASE_TAG)
print("工作目录  :", WORK)

# 各场景授权所用的 fixture。每一个都是仓库中自带的、真实降采样后的测序数据，
# 并与满足它的那个 reference 角色配对。
#
# 这里的 release 标签就是已发布数据集实际使用的标签，因此每一处选择都可以直接换成
# 真实拉取而无需改动标识 —— 见下文「在自己的环境里下载真实参考基因组」。
SCENARIOS = {
    "rrbs": {
        "mode": "RRBS",
        "accession": "SRR31480456",
        "references": {"primary": "hg19@GRCh37.p13-gencode-v19"},
    },
    "rnaseq": {
        "mode": "RNASEQ",
        "accession": "SRR018258",
        "references": {"primary": "hg38@GRCh38-gencode-v44"},
    },
    "bs-pdx": {
        "mode": "RRBS",
        "accession": "SRR36187610",
        "references": {
            "graft": "hg38@GRCh38-gencode-v44",
            "host": "mm10@GRCm38-gencode-M25",
        },
    },
    "rna-pdx": {
        "mode": "RNASEQ",
        "accession": "SRR30880970",
        "references": {
            "graft": "hg38@GRCh38-gencode-v44",
            "host": "mm10@GRCm38-gencode-M25",
        },
    },
}

print("\nscenarios:", ", ".join(SCENARIOS))

## 1. 真实运行 `otter-install` 安装器

传统生信流程最让人头疼的就是“配环境” —— 各种软件版本冲突、缺少系统底层库等。  
OTTER 提供了一个纯静态编译的安装器 `otter-install`，它能自动完成：
1. 下载所有核心工具的静态二进制文件（开箱即用，无需依赖系统复杂的动态链接库）；
2. 部署 Craftmake 工作流流程目录（Workflow Catalog）；
3. 调用自带的 `enva` 工具，自动求解并创建隔离的 Conda 运行环境（包含 FastQC, Bismark, Bowtie2, STAR 等生信软件）。

有两个实用参数值得了解：
- `-install-dir`：把安装路径指定在 Colab 当前会话目录内，便于随时清理；
- `-non-interactive`：自动接受默认配置，无需人工在终端按回车，非常适合在 Notebook 中无人值守执行。

In [ ]:
installer_url = (
    f"https://github.com/{REPO}/releases/{RELEASE_TAG}/download/"
    "otter-install-linux-amd64-static"
)
installer_path = WORK / "otter-install"

sh(f"curl -fsSL -o {installer_path} {installer_url}")
installer_path.chmod(0o755)
print(f"\n安装器: {installer_path} ({installer_path.stat().st_size:,} bytes)")

### 第一步：先看计划，再做操作 (`-dry-run`)

成熟的软件工具应该在进行大规模文件下载或环境写入前，让用户清楚知道“它到底打算做什么”。  
通过 `-dry-run` 参数，安装器只会打印出计划下载的软件清单、安装目标路径以及环境创建方案，而不会真正改动系统。

In [ ]:
# 赋值给变量，而不是让这条调用成为 cell 的最后一个表达式。Jupyter 会把 cell 末尾裸调用的
# 返回值以 repr 形式自动显示，从而把整段捕获到的 stdout 以一行转义文本再打印一遍，
# 把上面可读的输出全部淹没。
plan = sh(
    f"{installer_path} -dry-run -non-interactive "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    capture=True,
)

### 第二步：正式执行安装

现在开始真实安装！
- **关于环境创建时间**：如果选择完整创建 Conda 生信环境（默认 `SKIP_ENVS = False`），由于需要联网下载多个生信软件包，通常需要耗时 3~5 分钟。输出窗口会实时打印进度，请耐心等待。
- ⚡ **小白极速体验技巧**：如果你只是想快速浏览 OTTER 的命令行交互、体验项目创建和快照生成，不想等待几分钟的环境下载，可以将下方的 `SKIP_ENVS` 修改为 `True`（仅下载预编译工具二进制，约十余秒即可完成）。

In [ ]:
SKIP_ENVS = False  # 设为 True 可跳过数分钟的环境创建

skip_flag = "-skip-envs" if SKIP_ENVS else ""

# 用 stream=True 而不是 capture=True：这一步会下载一整套生信软件栈，可能耗时数分钟。
# 用 capture 会在结束前什么都不显示，看起来就像卡住了。赋值而不是裸调用，
# 以免 Jupyter 再额外显示一次它的 repr。
install = sh(
    f"{installer_path} -non-interactive {skip_flag} "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    stream=True,
)

### 第三步：检查安装成果与工具版本

安装完成后，我们把新安装的工具目录加入系统的环境变量 `$PATH` 中，然后检查各个工具是否能正常汇报版本。

In [ ]:
os.environ["PATH"] = f"{INSTALL_DIR}:{os.environ['PATH']}"

print("=== 已安装的二进制 ===")
for binary in sorted(INSTALL_DIR.iterdir()):
    if binary.is_file() and os.access(binary, os.X_OK):
        print(f"  {binary.name}")

print("\n=== otter 与 craftmake 版本 ===")
sh("otter --version", capture=True)
sh("craftmake --version", capture=True)

print("=== 安装器部署的 workflow catalog ===")
catalog = INSTALL_DIR / "workflows"
print(f"  {catalog}: {sorted(p.name for p in catalog.iterdir()) if catalog.is_dir() else 'MISSING'}")

In [ ]:
print("=== enva 环境 ===")
if SKIP_ENVS:
    print("  已跳过（-skip-envs）")
else:
    sh("enva list", capture=True, check=False)

    # 证明 otter-core 不只是「被列出」，而是真的可用：这正是
    # 「环境目录存在」与「环境能用」的区别。
    print("\n=== otter-core 真的能运行工具吗？ ===")
    result = sh("enva run otter-core -- fastqc --version", capture=True, check=False)
    print("  可用:", result.returncode == 0)

## 2. 获取测试测序数据与辅助演练工具

接下来我们需要真实的测序数据来进行后续的项目构建演练。我们对官方 GitHub 仓库进行浅克隆（Shallow Clone，只拉取最新代码，不下载历史版本，几秒钟即可完成）：
- **真实的测试数据**：获取 `testdata/` 目录下的轻量 FASTQ 文件（保留了真实的测序格式与质量字符，但仅截取了极少数量的数据，便于秒级测试）；
- **`stub-registry` 辅助工具**：用于在本地瞬间写出符合工业级规范的模拟参考基因组。编译该辅助工具需要 Go 语言环境，下方单元格会自动检测并在缺失时自动配置。

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    # 浅克隆且不带 submodule：这次克隆只为了 fixture 和演练脚本，
    # 工具链来自 release。
    sh(
        f"git clone --depth 1 --no-recurse-submodules "
        f"https://github.com/{REPO}.git {REPO_DIR}",
        stream=True,
    )
else:
    print(f"复用已有克隆：{REPO_DIR}")

print()
sh(f"git -C {REPO_DIR} log --oneline -1", capture=True)

fixture_root = REPO_DIR / "testdata/gate6/craftmake-downsample-20260906/fastq"
print(f"\nfixture: {fixture_root}")
print("  样本编号:", sorted(p.name for p in fixture_root.iterdir()))

In [ ]:
# stub-registry 是本 notebook 唯一需要编译的程序。
if shutil.which("go") is None:
    print("未找到 Go，正在安装（几秒钟）。")
    sh("apt-get -qq update && apt-get -qq install -y golang-go", check=False)

go_path = shutil.which("go")
print("go:", go_path or "仍然缺少")
if go_path:
    # 赋值而不是裸调用，以免 Jupyter 再额外显示一次它的 repr。
    go_version = sh("go version", capture=True)

In [ ]:
stub_registry = INSTALL_DIR / "stub-registry"

if go_path is None:
    raise RuntimeError(
        "stub-registry 需要 Go 工具链。请安装 Go，或预先编译好该二进制并放到 "
        f"{stub_registry}。"
    )

sh(f"cd {REPO_DIR} && go build -o {stub_registry} ./internal/e2esupport/cmd/stub-registry")
print(f"\nstub-registry: {stub_registry.stat().st_size:,} bytes")

## 3. 认识并模拟参考基因组注册表（Reference Registry）

### 什么是参考基因组与算法索引（Index）？
在进行高通量测序分析前，计算机必须手握一本标准“字典”——即参考基因组（如人类 hg38、hg19，小鼠 mm10）。为了快速在几十亿个碱基中定位测序片段，比对软件（STAR、Bowtie2、Bismark 等）必须预先对基因组序列构建庞大而复杂的二叉树或后缀数组索引（Index）。

### 为什么 OTTER 采用“注册表”设计？
在传统生信流程中，参考基因组路径往往散落在各种脚本里，很容易出现“误改文件”、“版本对不上”、“多人协作时哈希不一致”的学术复现难题。  
OTTER 采用了工业级的**集中式注册表（Reference Registry）**架构：
1. **全局不可变**：每个参考基因组发布为一个独立的 release 目录，严格只读（chmod 0444/0555）；
2. **加密指纹清单**：目录内的每一个文件都记录在 `manifest.json` 与 `checksums.sha256` 中，拥有唯一的 SHA-256 哈希值；
3. **分析项目与参考数据解耦**：分析项目内只记录参考基因组的逻辑代号与哈希指纹（如 `hg38@GRCh38-gencode-v44`），数据文件留在注册表中统一引用，绝不重复拷贝进用户目录。

### 为什么在这个 Colab 里使用“模拟”而不是直接下载？
- **存储空间瓶颈**：真实的哺乳动物参考基因组及全套索引极其庞大（例如仅人类 hg38 的 STAR 索引解压后就达 30 GB 左右，一套完整基因组文件高达 50~80 GB）。免费 Colab 虚拟机的临时磁盘只有约 100 GB，一旦下载一两个真实基因组，磁盘就会被撑爆，后续分析完全无法进行。
- **验证核心逻辑足矣**：通过 `stub-registry`，我们能在 1 秒钟内生成一个**目录结构、权限设定、哈希清单 100% 真实**的模拟注册表，足以完整驱动 OTTER 的所有配置与解析校验逻辑！

---

### 💡 在自己的真实服务器中，如何下载和使用真正的参考基因组？
当你回到具有大容量磁盘的工作站、实验室服务器或 HPC 超算集群时，OTTER 提供了三套清晰可靠的真实数据获取途径：

#### 方案 A：一键拉取预编译好的完整索引包（官方推荐）
OTTER 团队在 Hugging Face 平台上托管了预先构建好的高质量全套参考基因组包，你完全不需要自己开几十核 CPU 耗费数小时去慢慢构建：
```text
https://huggingface.co/datasets/fallingstar10/xdxtools-genomes
官方公开数据集已包含：
  - hg19@GRCh37.p13-gencode-v19  (人类 hg19 / GRCh37)
  - hg38@GRCh38-gencode-v44      (人类 hg38 / GRCh38)
  - mm10@GRCm38-gencode-M25      (小鼠 mm10 / GRCm38)
  - mm39@GRCm39-gencode-vM39     (小鼠 mm39 / GRCm39)
  - mm9@NCBIM37-gencode-M1       (小鼠 mm9 / NCBIM37)
```
在你的真实环境中，只需两条环境变量加一条命令即可自动拉取、解压并配置好：
```bash
# 1. 指定你想拉取的参考基因组版本（支持逗号分隔多个）
export OTTER_REFERENCE_FETCH_RELEASES="hg38@GRCh38-gencode-v44"

# 2. 指定你服务器上的参考注册表存储路径（例如共享存储挂载点）
export OTTER_REFERENCE_FETCH_REGISTRY_ROOT=/shared/otter/references

# 3. 运行自动拉取（自动完成下载、完整性校验、按规范解包）
otter-install -reference-fetch -non-interactive
```
*💡 贴心提示：如果你的硬盘较小，用不到某些特别大的索引（如 30G 的 STAR），可以通过环境变量 `OTTER_REFERENCE_FETCH_ASSETS="bismark,bowtie2,fasta,annotations"` 定向排除不需要的算法索引。在国内网络访问受限时，可通过 `OTTER_REFERENCE_FETCH_BASE_URL` 指定镜像源。*

#### 方案 B：本地从零构建（适合小众研究物种或定制基因组）
如果你研究的是拟南芥、线虫、酵母或特殊自测组装的基因组，可以使用 `otter reference build` 传入原始 FASTA 和 GTF 注释文件，工具链会自动调用 samtools、Bismark、Bowtie2、STAR 一键完成索引构建并封装备案为标准注册表：
```bash
otter reference build \n  --id my_species \n  --release custom-v1 \n  --organism "Danio rerio" \n  --assembly "GRCz11" \n  --fasta /path/to/genome.fa.gz \n  --gtf /path/to/genes.gtf.gz
```

#### 方案 C：网页浏览器直连下载
直接打开 Hugging Face 网页界面 `https://huggingface.co/datasets/fallingstar10/xdxtools-genomes`，手动下载对应物种压缩包，解压放置在 `<注册表根目录>/genomes/<物种id>/<版本release>/` 目录下即可。

---

### 在分析项目中无缝使用
无论走上述哪种途径获取的参考基因组，后续在项目中调用的方式与本教程完全相同 —— 只需要在命令行中提供 `--reference-primary <物种代号>@<版本标签>` 即可，OTTER 会自动定位并挂载对应的索引！

In [ ]:
# 各场景会选用的全部参考基因组。PDX 场景会同时声明 graft 与 host，
# 因此两个物种都需要。
#
# 这些必须与上文 SCENARIOS 中的选择一致：场景只能选用注册表中确实存在的
# release，所以这两个列表是同一份契约。这里的标签就是已发布数据集实际使用的
# 标签，读者因此可以把模拟注册表换成真实拉取，而无需改动选择。
REFERENCES = [
    {
        "id": "hg19",
        "release": "GRCh37.p13-gencode-v19",
        "organism": "Homo sapiens",
        "assembly": "GRCh37.p13",
        "aliases": "hg19,human,grch37",
    },
    {
        "id": "hg38",
        "release": "GRCh38-gencode-v44",
        "organism": "Homo sapiens",
        "assembly": "GRCh38",
        "aliases": "hg38,human,grch38",
    },
    {
        "id": "mm10",
        "release": "GRCm38-gencode-M25",
        "organism": "Mus musculus",
        "assembly": "GRCm38",
        "aliases": "mm10,mouse,grcm38",
    },
]

for reference in REFERENCES:
    sh(
        f"{stub_registry} --registry-root {REGISTRY} "
        f"--id {reference['id']} --release {reference['release']} "
        f"--organism '{reference['organism']}' --assembly {reference['assembly']} "
        f"--alias {reference['aliases']}"
    )
    print()

In [ ]:
import json

print("=== 注册表目录结构 ===")
sh(f"find {REGISTRY} -maxdepth 4 -mindepth 3 | sort", capture=True)

release_dir = REGISTRY / "genomes/hg19/GRCh37.p13-gencode-v19"
print("=== reference.yaml（release 的身份）===")
print("\n".join(
    f"  {line}" for line in (release_dir / "reference.yaml").read_text().splitlines()[:14]
))

manifest = json.loads((release_dir / "manifest.json").read_text())
print(f"\n=== manifest.json：{len(manifest)} 条记录 ===")
for entry in manifest[:5]:
    print(f"  {entry['path']}")
print(f"  ... 另有 {max(0, len(manifest) - 5)} 条")

# release 契约由 reference.yaml + manifest.json + checksums.sha256 组成。这里只报告
# fixture 实际写出了哪几个，而不是假定三个都在：stub 与生产发布器是两条独立的代码路径，
# 一个断言了某条实现输出的 notebook，会在另一条上直接崩掉。
print("\n=== fixture 写出的契约文件 ===")
contract_files = ["reference.yaml", "manifest.json", "checksums.sha256"]
for name in contract_files:
    path = release_dir / name
    if path.is_file():
        print(f"  {name:<20} {path.stat().st_size:>6} bytes")
    else:
        print(f"  {name:<20} 缺失")

checksums_path = release_dir / "checksums.sha256"
if checksums_path.is_file():
    print("\n=== checksums.sha256 覆盖了 release 身份 ===")
    print("\n".join(
        f"  {line}"
        for line in checksums_path.read_text().splitlines()[:4]
    ))
    print("\n  按其中列出的每个文件逐一校验：")
    # 赋值而不是裸调用：cell 末尾的裸调用会被自动显示为 repr，把 stdout
    # 以一行转义文本再打印一遍。
    checksum_check = sh(f"cd {release_dir} && sha256sum -c checksums.sha256 | tail -3", capture=True)
else:
    print(
        "\nchecksums.sha256 不存在，因此无法在这里演示 sha256sum -c。"
        "\n该文件是 release 契约的一部分，它缺失说明 fixture 有缺口："
        "\n应当如实报告，而不是把该 release 当作完整的。"
    )

## 4. 使用 `otter build` 一步完成项目创建与参数绑定

在传统的生信分析中，新手往往需要：
1. 手动创建复杂的文件夹，从各处拷贝脚本和 Snakefile；
2. 编写长篇大论且极易因空格或制表符缩进而报错的 YAML 配置文件；
3. 手动核对 FASTQ 样本命名，自己写配对列表；
4. 核对基因组索引的物理路径是否拼写正确。

而在 OTTER 中，新增的 **`otter build`** 子命令把整个前置步骤变成了**一条纯粹的自动化命令**：
- **智能样本扫描**：自动扫描 FASTQ 目录，识别 R1 和 R2 双端测序对，核对样本元数据（pdata），并自动生成接头序列信息；
- **加密指纹锁定**：自动从参考注册表中检索对应的基因组，将不可变的 SHA-256 哈希值写入 `references.lock.yaml`；
- **规范项目结构生成**：自动创建规范的项目配置文件 `project.yaml` 及样本清单 `samples.tsv`；
- **生成任务快照 (Run Snapshot)**：自动生成一个全局唯一的、带时间戳的只读任务快照 `run.yaml`。

接下来，我们在下面的单元格中，对四大分析场景（RRBS、RNA-seq、BS-PDX、RNA-PDX）分别执行一次 `otter build`！

In [ ]:
def stage_inputs(scenario, accession):
    """把一对 fixture 按 `otter create` 期望的命名复制好，并写出对应的 pdata。"""
    project_dir = PROJECTS / scenario
    fastq_dir = project_dir / "fastq"
    fastq_dir.mkdir(parents=True, exist_ok=True)

    source = fixture_root / accession
    shutil.copyfile(source / "R1.fastq.gz", fastq_dir / f"{accession}_R1.fastq.gz")
    shutil.copyfile(source / "R2.fastq.gz", fastq_dir / f"{accession}_R2.fastq.gz")

    pdata = project_dir / "pdata.csv"
    pdata.write_text(
        "sampleid,inline_barcode_sequence,condition\n" f"{accession},,case\n"
    )
    return project_dir, fastq_dir, pdata


def reference_arguments(references):
    """渲染角色参数。primary 是单角色；PDX 则是 graft 加 host。"""
    if "primary" in references:
        return f"--reference-primary {references['primary']}"
    return (
        f"--reference-graft {references['graft']} "
        f"--reference-host {references['host']}"
    )


snapshots = {}

for scenario, spec in SCENARIOS.items():
    print("=" * 72)
    print(f"场景: {scenario}  (mode={spec['mode']}, fixture={spec['accession']})")
    print("=" * 72)

    project_dir, fastq_dir, pdata = stage_inputs(scenario, spec["accession"])
    references = reference_arguments(spec["references"])

    result = sh(
        f"otter build --project-root {project_dir} "
        f"--fastq {fastq_dir} --pdata {pdata} --mode {spec['mode']} "
        f"--jobid {scenario} "
        f"--reference-root {REGISTRY} {references} --backend local",
        capture=True,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(f"otter build failed for {scenario} (exit {result.returncode})")

    # `build` 会把 snapshot 路径单独打印在最后一行。
    snapshot_path = Path(result.stdout.strip().splitlines()[-1].strip())
    assert snapshot_path.name == "run.yaml", f"unexpected snapshot path: {snapshot_path}"
    assert snapshot_path.exists(), f"reported snapshot does not exist: {snapshot_path}"
    snapshots[scenario] = snapshot_path

    for artifact in ("project.yaml", "samples.tsv", "references.lock.yaml"):
        assert (project_dir / artifact).exists(), f"{scenario} is missing {artifact}"

    print(f"  snapshot: {snapshot_path.relative_to(WORK)}")
    print()

print(f"已授权 {len(snapshots)} 个场景: {', '.join(snapshots)}")

### 探查 `otter build` 产出的项目目录

我们以刚才构建完成的 `rrbs` 场景为例，看看生成的项目长什么样：
1. **`project.yaml`**：记录清晰的高级分析目标（如分析类型为 RRBS，执行引擎为 Craftmake）；
2. **`samples.tsv`**：记录每个样本对应的测序文件相对路径；
3. **`references.lock.yaml`**：将当前项目引用的参考基因组哈希指纹锁定；
4. **`workflows/rules/`**：核心分析规则完整保存在项目内，避免依赖外部全局路径，而项目根目录保持干净清爽。

In [ ]:
sample_project = PROJECTS / "rrbs"

print("=== 项目目录结构 ===")
sh(f"find {sample_project} -maxdepth 2 -type d | sort", capture=True)

print("=== project.yaml ===")
print("\n".join(
    f"  {line}" for line in (sample_project / "project.yaml").read_text().splitlines()
))

print("\n=== samples.tsv（路径相对项目根目录）===")
print("\n".join(
    f"  {line}" for line in (sample_project / "samples.tsv").read_text().splitlines()
))

print("\n=== references.lock.yaml（被锁定的 digest）===")
print("\n".join(
    f"  {line}"
    for line in (sample_project / "references.lock.yaml").read_text().splitlines()
))

print("\n=== 规则固定在 workflows/ 下，而不是项目根目录 ===")
print("  workflows/rules 存在:", (sample_project / "workflows/rules").is_dir())
print("  项目根 rules/ 存在:", (sample_project / "rules").exists())

## 5. 执行 `--dry-run` 任务预演：验证所有阶段能否顺利规划

### 什么是 `--dry-run`（试跑演练）？
在真实的科研计算中，完整运行一个生信流程可能需要几个小时。如果由于“参数冲突”或者“第三阶段的输入找不到”而在跑了 5 个小时之后突然崩溃，会极大地浪费宝贵的计算资源和科研时间。

因此，OTTER 的 `--dry-run` 提供了**极速的“流程彩排”能力**：
- 严格读取不可变的快照 `run.yaml`；
- 调用执行引擎 Craftmake，按照工作流目录依次解析该流程的每一个阶段：
  - `step1`：测序读段质控与接头过滤（QC & Trimming）
  - `step2`：序列对齐比对（Alignment）
  - `step2-check`：比对产物完整性校验
  - `step3`：甲基化提取或表达量矩阵定量（Quantification）
  - `step3-check`：定量结果校验
  - `publish`：最终分析产物归档发布
- 编译并生成完整的**任务依赖关系图（DAG）**，逐一验证每一个任务的前后衔接。

**在这个过程中，不会执行具体的消耗大量算力的生信算法，不会修改测序数据**，专门用于在提交昂贵的计算任务前确认 100% 正确！

In [ ]:
import json

# 显式指定而不是依赖 PATH：解析器会退回做 PATH 查找，而一个会调整自身环境的
# notebook 不应该因此失败。
craftmake_binary = INSTALL_DIR / "craftmake"
catalog = INSTALL_DIR / "workflows"


def phases_for(workflow):
    """从已部署的 catalog 中按顺序列出某个 workflow 已发布的 phase。

    从 catalog 读取而不是写死：各 workflow 的 phase 集合不同（RNA-seq 没有 step3），
    写死的列表会在新增 phase 的那一刻悄然失效。
    """
    order = {"step1": 0, "step2": 1, "step2-check": 2, "step3": 3, "step3-check": 4, "publish": 5}
    names = [p.stem for p in (catalog / workflow).glob("*.yaml")]
    return sorted(names, key=lambda name: (order.get(name, 99), name))


def plan_phase(scenario, snapshot_path, phase):
    """规划一个 phase 并返回其 envelope。

    用 echo=False，因为一个 plan envelope 是几百 KB 的 JSON：打印它会淹没这个 cell
    真正要产出的汇总。真正有用的是任务数量，它会在下面打印。
    """
    result = sh(
        f"otter run --config {snapshot_path} "
        f"--executor craftmake --phase {phase} "
        f"--dry-run --foreground "
        f"--craftmake-binary {craftmake_binary} --catalog {catalog}",
        echo=False,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"dry run failed for {scenario} {phase} (exit {result.returncode})")
    return json.loads(result.stdout)


# 每个 phase 都会被规划，而不只是 step1。一个 run 解析出来的是多阶段 workflow，
# 只规划一个 phase 只能证明那一个 phase；后续 phase 会消费前面 phase 的产物，
# 解析问题恰恰容易在那里暴露。
plan_results = {}
total_tasks = 0

for scenario, snapshot_path in snapshots.items():
    print("=" * 72)
    print(f"规划: {scenario}")
    print("=" * 72)

    # 从 snapshot 自身的 step1 计划里取得 workflow 名，这样 phase 列表来自
    # 这次 run 实际选中的东西，而不是某个命名约定。
    first_envelope = plan_phase(scenario, snapshot_path, "step1")
    workflow = first_envelope["data"]["workflow"]
    phase_names = phases_for(workflow)
    print(f"  workflow: {workflow}")
    print(f"  phases  : {', '.join(phase_names)}")
    print()

    scenario_plans = {}
    for phase in phase_names:
        envelope = (
            first_envelope
            if phase == "step1"
            else plan_phase(scenario, snapshot_path, phase)
        )

        assert envelope.get("command") == "plan", envelope.get("command")
        assert envelope.get("ok") is True, envelope
        tasks = envelope.get("data", {}).get("tasks", [])
        assert tasks, f"{scenario} {phase} returned no tasks"

        scenario_plans[phase] = len(tasks)
        total_tasks += len(tasks)
        print(f"    {phase:<12} {len(tasks):>3} 个任务")
    print()

    plan_results[scenario] = {"workflow": workflow, "phases": scenario_plans}

print(f"已规划 {len(plan_results)} 个场景的全部 phase，共 {total_tasks} 个任务")
print("没有执行任何任务")

## 6. 全流程演练总结

恭喜！到这里，你已经成功演练了从软件安装、注册表模拟、项目自动化构建到全阶段任务预演的完整生信流程。  
下方的汇总表清晰列出了四大场景所调用的对应工作流引擎，以及各阶段成功规划出的任务数量。

In [ ]:
import unicodedata
from datetime import datetime, timezone

print("=" * 72)
print("OTTER Colab 演练 —— 汇总")
print("=" * 72)
print(f"完成时间  : {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} UTC")
print(f"仓库      : {REPO}")
print(f"release   : {RELEASE_TAG}")
print(f"环境      : {'已跳过' if SKIP_ENVS else 'otter-core（已创建）'}")
print()

# CJK glyphs are two columns wide, so pad by display width instead of by character count
# or the columns will not line up.
def pad(text, width, right=False):
    filler = " " * max(0, width - sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in text))
    return filler + text if right else text + filler

header = pad("场景", 10) + pad("mode", 8) + pad("fixture", 12) + pad("workflow", 17) + pad("phase", 6, True) + pad("任务", 6, True)
print(header)
print("-" * len(header))
for scenario, spec in SCENARIOS.items():
    result = plan_results[scenario]
    phase_tasks = result["phases"]
    print(
        pad(scenario, 10) + pad(spec['mode'], 8) + pad(spec['accession'], 12)
        + pad(result['workflow'], 17) + pad(str(len(phase_tasks)), 6, True)
        + pad(str(sum(phase_tasks.values())), 6, True)
    )

print()
print("每个场景都解析成了不可变 snapshot，每个 workflow 的每个 phase 都到达了")
print("Craftmake 的 planner。没有执行任何生信工具，也没有任何结果是科学结论。")
print()
print(f"{WORK} 下的产物：")
print(f"  {INSTALL_DIR}   已发布的二进制")
print(f"  {REGISTRY}      模拟的参考基因组注册表")
print(f"  {PROJECTS}      已授权项目与已解析的 run")

## 接下来可以探索什么？

### 1. 体验更严格的自动化全量测试脚本
本仓库内置了一个自动化演练脚本 `scripts/e2e/otter_e2e.sh`，包含 103 个阶段性严苛断言，会对比新旧两条执行路径、验证多种集群调度模式和报错拦截机制：
```bash
cd /content/otter-colab/repo
bash scripts/e2e/otter_e2e.sh \n  --otter            /content/otter-colab/bin/otter \n  --craftmake        /content/otter-colab/bin/craftmake \n  --stub-registry    /content/otter-colab/bin/stub-registry \n  --installer        /content/otter-colab/otter-install \n  --craftmake-catalog /content/otter-colab/bin/workflows
```

### 2. 在你自己的真实服务器上执行真实计算
当你在带有真实测序数据的 Linux 服务器或 HPC 超算上工作时，去掉 `--dry-run` 参数，OTTER 就会真正调起计算引擎开始极速比对与计算：
```bash
# 启动真实计算（例如执行第一阶段）
otter run --config /path/to/run.yaml --executor craftmake --phase step1 --backend local

# 查看后台任务运行状态或实时日志
otter task list
otter task logs <task-id> --follow
```

### 3. 查阅完整官方文档
想了解更多进阶主题（如 SLURM 超算集群配置、多样本批次效应校正、PDX 人鼠细胞污染分离算法等），欢迎查阅：
- [OTTER 官方用户手册](https://github.com/otterlab-bio/otter/blob/main/docs/manual/README.md)
- [参考基因组注册表技术规范](https://github.com/otterlab-bio/otter/blob/main/docs/manual/08-reference-migration.md)

---
*祝你的生信分析顺利开展！*